# DOJ Attorney Hiring: Postings, Hires, and the Shrinking Workforce

This notebook looks at three things for DOJ series 0905 (General Attorney):
1. **Headcount** — is the total attorney workforce actually smaller?
2. **Accessions** — how has actual hiring changed since January 2025?
3. **Postings** — are they still advertising for attorneys?

## Data sources

**OPM EHRI** (`impactproject/opm-ehri-data` on HuggingFace)  
Two datasets: monthly accessions (new hires) and monthly employment snapshots (headcount). Queried server-side via DuckDB's `hf://` URI — no local download.

**USAJobs** (R2 parquet mirror at `pub-317c58882ec04f329b63842c1eb65b0c.r2.dev`)  
Historical postings harmonized from both USAJobs APIs, stored as annual parquet files. Queried directly via DuckDB — no download.

## How the connection works

OPM's four-digit occupational series codes appear in both EHRI and USAJobs. Series `0905` is General Attorney — the primary attorney classification at DOJ. Filtering all three datasets to `agency = DOJ` and `series = 0905` puts them on the same population. There will be a lag between postings and hires (federal hiring typically takes 3–6 months), which we estimate from pre-2025 data.

In [ ]:
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 12,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

INAUGURATION = pd.Timestamp('2025-01-20')
R2 = 'https://pub-317c58882ec04f329b63842c1eb65b0c.r2.dev'
HF = 'hf://datasets/impactproject/opm-ehri-data'

conn = duckdb.connect()

## Helper: pick latest version per month

EHRI files have multiple versions (v1, v2, v3) as OPM updates data. We pick the highest version available for each month.

In [ ]:
def latest_version_urls(glob_pattern, prefix, year_min=2022):
    """Return one HF URL per YYYYMM — the highest version available."""
    all_files = conn.execute(f"SELECT file FROM glob('{glob_pattern}')").df()['file'].tolist()
    best = {}
    for f in all_files:
        name = f.split('/')[-1]           # e.g. accessions_202601_v1.parquet
        parts = name.replace('.parquet', '').split('_')
        yyyymm, ver = parts[1], int(parts[2][1:])
        if int(yyyymm[:4]) >= year_min:
            if yyyymm not in best or ver > best[yyyymm][1]:
                best[yyyymm] = (f, ver)
    return [v[0] for v in sorted(best.values(), key=lambda x: x[0])]

## Part 1: EHRI Accessions — Who Actually Got Hired

In [ ]:
acc_urls = latest_version_urls(f'{HF}/accessions/*.parquet', 'accessions')
url_list = ', '.join(f"'{u}'" for u in acc_urls)

acc_df = conn.execute(f"""
    SELECT 
        regexp_extract(filename, 'accessions_([0-9]{{6}})', 1) AS yyyymm,
        SUM(CAST(count AS INTEGER)) AS accessions
    FROM read_parquet([{url_list}], filename=true)
    WHERE agency ILIKE '%JUSTICE%'
      AND occupational_series_code = '0905'
    GROUP BY 1 ORDER BY 1
""").df()

acc_df['date'] = pd.to_datetime(acc_df['yyyymm'], format='%Y%m')
print(f"Accessions: {acc_df['date'].min().strftime('%Y-%m')} → {acc_df['date'].max().strftime('%Y-%m')}")
acc_df.tail(8)

## Part 2: EHRI Employment — Total Headcount

The employment snapshot shows how many DOJ 0905 attorneys are on payroll each month — the net result of hiring minus all types of attrition.

In [ ]:
emp_urls = latest_version_urls(f'{HF}/employment/*.parquet', 'employment')
emp_url_list = ', '.join(f"'{u}'" for u in emp_urls)

emp_df = conn.execute(f"""
    SELECT 
        regexp_extract(filename, 'employment_([0-9]{{6}})', 1) AS yyyymm,
        SUM(CAST(count AS INTEGER)) AS headcount
    FROM read_parquet([{emp_url_list}], filename=true)
    WHERE agency = 'DEPARTMENT OF JUSTICE'
      AND occupational_series_code = '0905'
    GROUP BY 1 ORDER BY 1
""").df()

emp_df['date'] = pd.to_datetime(emp_df['yyyymm'], format='%Y%m')
peak = emp_df.loc[emp_df['headcount'].idxmax()]
latest = emp_df.iloc[-1]
loss = int(peak.headcount - latest.headcount)
print(f"Peak: {int(peak.headcount):,} ({peak.date.strftime('%b %Y')})")
print(f"Latest: {int(latest.headcount):,} ({latest.date.strftime('%b %Y')})")
print(f"Net loss: {loss:,} attorneys ({loss/peak.headcount*100:.0f}%)")
emp_df.tail(8)

## Part 3: USAJobs Postings — What Was Advertised

The `occupationalSeries` field can contain multiple series separated by semicolons (e.g. `0905 - Attorney; 0950 - Paralegal`), so we filter with `LIKE '%0905%'`.

In [ ]:
# Historical files cover 2013–2026; current files overlap for 2024+
# Use DISTINCT on control number to deduplicate across the two sources
years = list(range(2019, 2027))
hist_urls = [f"'{R2}/data/historical_jobs_{y}.parquet'" for y in years]
curr_urls = [f"'{R2}/data/current_jobs_{y}.parquet'" for y in [2024, 2025, 2026]]
all_posting_urls = ', '.join(hist_urls + curr_urls)

post_df = conn.execute(f"""
    SELECT 
        LEFT(openDate, 7) AS month,
        COUNT(DISTINCT usajobsControlNumber) AS postings
    FROM read_parquet([{all_posting_urls}])
    WHERE hiringDepartmentName = 'Department of Justice'
      AND occupationalSeries LIKE '%0905%'
      AND openDate IS NOT NULL
    GROUP BY 1 ORDER BY 1
""").df()

post_df['date'] = pd.to_datetime(post_df['month'] + '-01')
post_df = post_df[post_df['date'] >= '2022-01-01'].reset_index(drop=True)
print(f"Postings: {post_df['month'].min()} → {post_df['month'].max()}")
post_df.tail(8)

## Part 4: Estimating the Posting-to-Hire Lag

Before looking at the post-inauguration period, we estimate the normal posting-to-hire delay using pre-2025 data only.

In [ ]:
combined = pd.merge(acc_df[['date', 'accessions']], post_df[['date', 'postings']], on='date', how='outer')
combined = combined.sort_values('date').fillna(0).reset_index(drop=True)

pre = combined[combined['date'] < '2025-01-01'].copy()

MAX_LAG = 12
correlations = []
for lag in range(MAX_LAG + 1):
    shifted = pre['postings'].shift(lag)
    valid = ~shifted.isna()
    r = np.corrcoef(shifted[valid], pre['accessions'][valid])[0, 1]
    correlations.append(r)

best_lag = int(np.argmax(correlations))
best_r = correlations[best_lag]

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(range(MAX_LAG + 1), correlations, color='#4393c3', alpha=0.85)
ax.axvline(best_lag, color='#d6604d', linewidth=2, linestyle='--')
ax.set_xlabel('Lag (months postings lead accessions)')
ax.set_ylabel('Pearson r')
ax.set_title(f'Cross-correlation (pre-2025 data only)\nBest lag = {best_lag} months (r = {best_r:.2f})',
             fontsize=11, fontweight='bold')
ax.set_xticks(range(MAX_LAG + 1))
plt.tight_layout()
plt.savefig('lag_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Best lag: {best_lag} months")

## Part 5: The Full Picture — Three Panels

In [ ]:
acc_df['smooth']  = acc_df['accessions'].rolling(3, center=True).mean()
post_df['smooth'] = post_df['postings'].rolling(3, center=True).mean()

fig, axes = plt.subplots(3, 1, figsize=(12, 11))
fig.suptitle('DOJ General Attorneys (Series 0905): The Full Picture',
             fontsize=14, fontweight='bold', y=1.01)

# ── Panel A: headcount ──
ax = axes[0]
ax.fill_between(emp_df['date'], emp_df['headcount'], color='#4393c3', alpha=0.25)
ax.plot(emp_df['date'], emp_df['headcount'], color='#2166ac', linewidth=2.5)
ax.axvline(INAUGURATION, color='#888', linewidth=1.2, linestyle='--')
ax.text(INAUGURATION + pd.Timedelta(days=12), emp_df['headcount'].max() * 0.986,
        'Jan 20, 2025', color='#555', fontsize=9)
ax.annotate(f"Peak: {int(peak.headcount):,}",
            xy=(peak.date, peak.headcount),
            xytext=(peak.date - pd.Timedelta(days=450), peak.headcount - 350),
            arrowprops=dict(arrowstyle='->', color='#555'), fontsize=9.5)
ax.annotate(f"{latest.date.strftime('%b %Y')}: {int(latest.headcount):,}\n(−{loss:,}, −{loss/peak.headcount*100:.0f}%)",
            xy=(latest.date, latest.headcount),
            xytext=(latest.date - pd.Timedelta(days=440), latest.headcount - 600),
            arrowprops=dict(arrowstyle='->', color='#555'), fontsize=9.5)
ax.set_ylabel('Attorneys on payroll')
ax.set_title('Headcount', fontsize=11, fontweight='bold')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

# ── Panel B: accessions ──
ax = axes[1]
ax.bar(acc_df['date'], acc_df['accessions'], width=25, color='#d6604d', alpha=0.4, label='Monthly')
ax.plot(acc_df['date'], acc_df['smooth'], color='#b2182b', linewidth=2, label='3-mo avg')
ax.axvline(INAUGURATION, color='#888', linewidth=1.2, linestyle='--')
ax.set_ylabel('New hires / month')
ax.set_title('New Hires (EHRI Accessions)', fontsize=11, fontweight='bold')
ax.legend(frameon=False, fontsize=9.5, loc='upper left')
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

# ── Panel C: postings ──
ax = axes[2]
ax.bar(post_df['date'], post_df['postings'], width=25, color='#4dac26', alpha=0.4, label='Monthly')
ax.plot(post_df['date'], post_df['smooth'], color='#1a7837', linewidth=2, label='3-mo avg')
ax.axvline(INAUGURATION, color='#888', linewidth=1.2, linestyle='--')
ax.set_ylabel('Postings / month')
ax.set_title('USAJobs Postings', fontsize=11, fontweight='bold')
ax.legend(frameon=False, fontsize=9.5, loc='upper left')
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.tight_layout()
plt.savefig('doj_full_picture.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: doj_full_picture.png')

## Part 6: LinkedIn Chart — The Headcount Story

In [ ]:
fig2, ax2 = plt.subplots(figsize=(10, 5))
ax2.fill_between(emp_df['date'], emp_df['headcount'], color='#4393c3', alpha=0.2)
ax2.plot(emp_df['date'], emp_df['headcount'], color='#2166ac', linewidth=2.5)
ax2.axvline(INAUGURATION, color='#c0392b', linewidth=1.5, linestyle='--')
ax2.text(INAUGURATION + pd.Timedelta(days=14), emp_df['headcount'].max() * 0.998,
         'Inauguration\nJan 20, 2025', color='#c0392b', fontsize=9.5, va='top')
ax2.set_title(
    f'DOJ Has Lost {loss:,} of Its Attorneys\n'
    f'Series 0905 Headcount, {emp_df["date"].min().strftime("%b %Y")} – {latest.date.strftime("%b %Y")}',
    fontsize=13, fontweight='bold')
ax2.set_ylabel('Attorneys on federal payroll')
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax2.xaxis.set_major_locator(mdates.MonthLocator(interval=4))
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.xticks(rotation=30, ha='right')
ax2.annotate('', xy=(latest.date, latest.headcount), xytext=(peak.date, peak.headcount),
             arrowprops=dict(arrowstyle='<->', color='#c0392b', lw=2))
ax2.text(peak.date + pd.Timedelta(days=60), (peak.headcount + latest.headcount) / 2,
         f'−{loss:,} attorneys\n(−{loss/peak.headcount*100:.0f}%)',
         color='#c0392b', fontsize=11, fontweight='bold')
ax2.text(0.02, 0.04, 'Source: OPM EHRI via impactproject/opm-ehri-data (HuggingFace)',
         transform=ax2.transAxes, fontsize=8, color='#888')
plt.tight_layout()
plt.savefig('doj_headcount_linkedin.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: doj_headcount_linkedin.png')

## What the data shows

**The workforce is dramatically smaller.** DOJ peaked at ~13,130 series-0905 attorneys in early 2024. As of April 2026 (latest data), there are ~10,259 — a loss of ~2,871 attorneys (~22%). The October 2025 employment drop of ~700 in a single month is consistent with a RIF event. Headcount is still declining.

**The early-2025 hiring freeze was real.** New hires averaged ~27/month in Feb–Jun 2025, versus ~75/month for the same period in 2024. Postings hit their lowest point in the full dataset in January 2025 (12 postings).

**Hiring and postings surged in 2026 — but it's not enough.** By early 2026, monthly postings are the highest in the 2018–2026 dataset. Accessions have returned to pre-DOGE levels. But total headcount is still declining, which means the losses from separations and firings are outpacing the new hires.

## Notes

- EHRI data lags 1–3 months; most recent months may be incomplete
- Some DOJ positions fill outside USAJobs (Schedule A, transfers, honors program)
- Series 0905 spans entry-level to senior litigators — no further breakdown here
- All EHRI data is anonymous aggregate counts; no individual-level information

**Sources**  
- EHRI: [`impactproject/opm-ehri-data`](https://huggingface.co/datasets/impactproject/opm-ehri-data) — queried via DuckDB `hf://`, no download  
- USAJobs: R2 mirror at `pub-317c58882ec04f329b63842c1eb65b0c.r2.dev` — queried via DuckDB, no download  
- Methodology reference: [`abigailhaddad/federal-data-guide`](https://github.com/abigailhaddad/federal-data-guide)